In [1]:
import os
import sys


for _ in range(1):
    os.chdir("..")


os.getcwd()

'/Users/dananufrieva/MCS/six-00-twenty-search/stg'

In [2]:
from importlib import reload

import posting_list
import search_engine

reload(posting_list)
reload(search_engine)

<module 'search_engine' from '/Users/dananufrieva/MCS/six-00-twenty-search/stg/search_engine.py'>

# PostingList

In [3]:
from posting_list import BasePostingList, PostingList, AntiPostingList

# Тестируем базовые структуры данных
print("=== Тестирование PostingList ===")
pl = PostingList([1, 3, 5, 7, 10])
print(f"Cost: {pl.cost}")
print(f"next(): {pl.next()}")  # 1
print(f"next(): {pl.next()}")  # 3
print(f"advance(6): {pl.advance(6)}")  # 7
print(f"next(): {pl.next()}")  # 10
print(f"next(): {pl.next()}")  # None

=== Тестирование PostingList ===
Cost: 5
next(): 1
next(): 3
advance(6): 7
next(): 7
next(): 10


In [4]:
BasePostingList?

Init signature: BasePostingList(doc_ids: List[int], segment_size: Optional[int] = None)
Docstring:      Abstract base class for posting lists
File:           ~/MCS/six-00-twenty-search/stg/posting_list.py
Type:           ABCMeta
Subclasses:     PostingList, AntiPostingList

In [5]:
PostingList([1]).cost

1

In [7]:
AntiPostingList([1], 100).cost

99

# SearchEngine.execute

## SearchEngine.execute_or

In [8]:
from search_engine import SearchEngine

SearchEngine.execute_or?

Signature: SearchEngine.execute_or(pls: List[posting_list.PostingList]) -> posting_list.PostingList
Docstring: Execute OR operation through merging with skipping duplicates
File:      ~/MCS/six-00-twenty-search/stg/search_engine.py
Type:      function

In [9]:
# Тесты для execute_or (с сортированными входными данными)


print("=== SearchEngine.execute_or ===")
# Пустые списки
result = SearchEngine.execute_or([])
assert result.doc_ids == []
assert result.cost == 0

# Один список
pl = PostingList([1, 3, 5])
result = SearchEngine.execute_or([pl])
assert result.doc_ids == [1, 3, 5]

# Два непересекающихся списка
pl1 = PostingList([1, 3, 5])
pl2 = PostingList([2, 4, 6])
result = SearchEngine.execute_or([pl1, pl2])
assert result.doc_ids == [1, 2, 3, 4, 5, 6]

# Два пересекающихся списка
pl1 = PostingList([1, 3, 5, 7])
pl2 = PostingList([3, 5, 8, 9])
result = SearchEngine.execute_or([pl1, pl2])
assert result.doc_ids == [1, 3, 5, 7, 8, 9]

# Три списка с дубликатами
pl1 = PostingList([1, 2, 5])
pl2 = PostingList([2, 3, 5, 7])
pl3 = PostingList([1, 5, 8])
result = SearchEngine.execute_or([pl1, pl2, pl3])
assert result.doc_ids == [1, 2, 3, 5, 7, 8]

# С пустыми постинг-листами
pl1 = PostingList([])
pl2 = PostingList([1, 2, 3])
pl3 = PostingList([])
result = SearchEngine.execute_or([pl1, pl2, pl3])
assert result.doc_ids == [1, 2, 3]

# Все списки пустые
pl1 = PostingList([])
pl2 = PostingList([])
result = SearchEngine.execute_or([pl1, pl2])
assert result.doc_ids == []

# Большие числа
pl1 = PostingList([100, 200, 300])
pl2 = PostingList([150, 200, 250])
result = SearchEngine.execute_or([pl1, pl2])
assert result.doc_ids == [100, 150, 200, 250, 300]

# Списки из одного элемента
pl1 = PostingList([42])
pl2 = PostingList([42])
pl3 = PostingList([100])
result = SearchEngine.execute_or([pl1, pl2, pl3])
assert result.doc_ids == [42, 100]

# Сортированные входные данные
pl1 = PostingList([1, 3, 5])
pl2 = PostingList([2, 4, 6])
result = SearchEngine.execute_or([pl1, pl2])
assert result.doc_ids == [1, 2, 3, 4, 5, 6]

# Много дубликатов (уже отсортированные)
pl1 = PostingList([1, 2])
pl2 = PostingList([2, 3])
result = SearchEngine.execute_or([pl1, pl2])
assert result.doc_ids == [1, 2, 3]

# Один элемент в нескольких списках
pl1 = PostingList([10])
pl2 = PostingList([10])
pl3 = PostingList([10])
result = SearchEngine.execute_or([pl1, pl2, pl3])
assert result.doc_ids == [10]

# Разные размеры списков
pl1 = PostingList([1])
pl2 = PostingList([1, 2, 3, 4, 5])
pl3 = PostingList([3, 6])
result = SearchEngine.execute_or([pl1, pl2, pl3])
assert result.doc_ids == [1, 2, 3, 4, 5, 6]

# Граничные значения
pl1 = PostingList([0])
pl2 = PostingList([999999])
result = SearchEngine.execute_or([pl1, pl2])
assert result.doc_ids == [0, 999999]

# Последовательные числа
pl1 = PostingList([1, 2, 3])
pl2 = PostingList([4, 5, 6])
pl3 = PostingList([7, 8, 9])
result = SearchEngine.execute_or([pl1, pl2, pl3])
assert result.doc_ids == [1, 2, 3, 4, 5, 6, 7, 8, 9]

# Частичное перекрытие
pl1 = PostingList([1, 2, 3, 4])
pl2 = PostingList([3, 4, 5, 6])
pl3 = PostingList([5, 6, 7, 8])
result = SearchEngine.execute_or([pl1, pl2, pl3])
assert result.doc_ids == [1, 2, 3, 4, 5, 6, 7, 8]

# Один большой список и один маленький
pl1 = PostingList([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
pl2 = PostingList([5])
result = SearchEngine.execute_or([pl1, pl2])
assert result.doc_ids == [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# Все одинаковые списки
pl1 = PostingList([1, 2, 3])
pl2 = PostingList([1, 2, 3])
pl3 = PostingList([1, 2, 3])
result = SearchEngine.execute_or([pl1, pl2, pl3])
assert result.doc_ids == [1, 2, 3]

print("Все тесты прошли успешно!")

=== SearchEngine.execute_or ===
Все тесты прошли успешно!


## SearchEngine.execute_and

In [10]:
from search_engine import SearchEngine

SearchEngine.execute_and?

Signature: SearchEngine.execute_and(pls: List[posting_list.PostingList]) -> posting_list.PostingList
Docstring: Execute AND operation
File:      ~/MCS/six-00-twenty-search/stg/search_engine.py
Type:      function

In [11]:
# Тесты для execute_and

# Пустые списки
result = SearchEngine.execute_and([])
assert result.doc_ids == []

# Один список
pl = PostingList([1, 3, 5])
result = SearchEngine.execute_and([pl])
assert result.doc_ids == [1, 3, 5]

# Два непересекающихся списка
pl1 = PostingList([1, 3, 5])
pl2 = PostingList([2, 4, 6])
result = SearchEngine.execute_and([pl1, pl2])
assert result.doc_ids == []

# Два пересекающихся списка
pl1 = PostingList([1, 3, 5, 7])
pl2 = PostingList([3, 5, 8, 9])
result = SearchEngine.execute_and([pl1, pl2])
assert result.doc_ids == [3, 5]

# Три списка с пересечениями
pl1 = PostingList([1, 2, 5, 7])
pl2 = PostingList([2, 3, 5, 7])
pl3 = PostingList([1, 2, 5, 8])
result = SearchEngine.execute_and([pl1, pl2, pl3])
assert result.doc_ids == [2, 5]

# С пустыми постинг-листами
pl1 = PostingList([])
pl2 = PostingList([1, 2, 3])
pl3 = PostingList([2, 3, 4])
result = SearchEngine.execute_and([pl1, pl2, pl3])
assert result.doc_ids == []

# Все списки пустые
pl1 = PostingList([])
pl2 = PostingList([])
result = SearchEngine.execute_and([pl1, pl2])
assert result.doc_ids == []

# Один пустой список среди непустых
pl1 = PostingList([1, 2, 3])
pl2 = PostingList([])
pl3 = PostingList([2, 3, 4])
result = SearchEngine.execute_and([pl1, pl2, pl3])
assert result.doc_ids == []

# Большие числа
pl1 = PostingList([100, 200, 300, 400])
pl2 = PostingList([150, 200, 250, 300])
result = SearchEngine.execute_and([pl1, pl2])
assert result.doc_ids == [200, 300]

# Списки из одного элемента
pl1 = PostingList([42])
pl2 = PostingList([42])
pl3 = PostingList([42])
result = SearchEngine.execute_and([pl1, pl2, pl3])
assert result.doc_ids == [42]

# Списки из одного элемента без пересечения
pl1 = PostingList([42])
pl2 = PostingList([100])
result = SearchEngine.execute_and([pl1, pl2])
assert result.doc_ids == []

# Полное совпадение
pl1 = PostingList([1, 2, 3, 4, 5])
pl2 = PostingList([1, 2, 3, 4, 5])
result = SearchEngine.execute_and([pl1, pl2])
assert result.doc_ids == [1, 2, 3, 4, 5]

# Частичное совпадение в начале
pl1 = PostingList([1, 2, 3, 4, 5])
pl2 = PostingList([1, 2, 6, 7, 8])
result = SearchEngine.execute_and([pl1, pl2])
assert result.doc_ids == [1, 2]

# Частичное совпадение в конце
pl1 = PostingList([1, 2, 3, 4, 5])
pl2 = PostingList([0, 3, 4, 5])
result = SearchEngine.execute_and([pl1, pl2])
assert result.doc_ids == [3, 4, 5]

# Множественные пересечения
pl1 = PostingList([1, 3, 5, 7, 9])
pl2 = PostingList([2, 3, 5, 8, 9])
pl3 = PostingList([3, 4, 5, 9, 10])
result = SearchEngine.execute_and([pl1, pl2, pl3])
assert result.doc_ids == [3, 5, 9]

# Один элемент
pl1 = PostingList([5])
pl2 = PostingList([5])
pl3 = PostingList([5])
result = SearchEngine.execute_and([pl1, pl2, pl3])
assert result.doc_ids == [5]

# Разные размеры списков
pl1 = PostingList([1, 2, 3, 4, 5])
pl2 = PostingList([3, 5])
pl3 = PostingList([2, 3, 5, 7])
result = SearchEngine.execute_and([pl1, pl2, pl3])
assert result.doc_ids == [3, 5]

# Граничные значения
pl1 = PostingList([0, 999999])
pl2 = PostingList([0, 1000000])
result = SearchEngine.execute_and([pl1, pl2])
assert result.doc_ids == [0]

# Последовательные числа с одним общим
pl1 = PostingList([1, 2, 3])
pl2 = PostingList([3, 4, 5])
result = SearchEngine.execute_and([pl1, pl2])
assert result.doc_ids == [3]

# Все одинаковые списки
pl1 = PostingList([10, 20, 30])
pl2 = PostingList([10, 20, 30])
result = SearchEngine.execute_and([pl1, pl2])
assert result.doc_ids == [10, 20, 30]

# Пересечение в одной точке
pl1 = PostingList([1, 5, 10])
pl2 = PostingList([3, 5, 7])
pl3 = PostingList([2, 5, 8])
result = SearchEngine.execute_and([pl1, pl2, pl3])
assert result.doc_ids == [5]

# Нет пересечения
pl1 = PostingList([1, 2, 3])
pl2 = PostingList([4, 5, 6])
pl3 = PostingList([7, 8, 9])
result = SearchEngine.execute_and([pl1, pl2, pl3])
assert result.doc_ids == []

print("Все тесты для execute_and прошли успешно!")

Все тесты для execute_and прошли успешно!


## SearchEngine.execute_and_not



In [12]:
from search_engine import SearchEngine

SearchEngine.execute_and_not?

Signature:
SearchEngine.execute_and_not(
    pl: posting_list.PostingList,
    not_pl: posting_list.AntiPostingList,
) -> posting_list.PostingList
Docstring: Execute X AND (NOT Y) operation
File:      ~/MCS/six-00-twenty-search/stg/search_engine.py
Type:      function

In [13]:
SEGMENT_SIZE = 10_000

# Тест 1: Пустой позитивный список
print("Тест 1: Пустой позитивный список")
pos_pl = PostingList([])
neg_pl = AntiPostingList([1, 3, 5], SEGMENT_SIZE)
result = SearchEngine.execute_and_not(pos_pl, neg_pl)
assert result.doc_ids == [], f"Expected [], got {result.doc_ids}"
print("✓ Пустой позитивный список")

# Тест 2: Анти-постинг-лист пустой (cost = 0) - весь сегмент исключен
print("Тест 2: Анти-постинг-лист пустой")
all_docs = list(range(SEGMENT_SIZE))
pos_pl = PostingList([1, 3, 5])
neg_pl = AntiPostingList(all_docs, SEGMENT_SIZE)  # cost = 0
result = SearchEngine.execute_and_not(pos_pl, neg_pl)
assert result.doc_ids == [1, 3, 5], f"Expected [1, 3, 5], got {result.doc_ids}"
print("✓ Анти-постинг-лист пустой")

# Тест 3: Нет пересечений - позитивный список не пересекается с анти-листом
print("Тест 3: Нет пересечений")
pos_pl = PostingList([1, 3, 5])
neg_pl = AntiPostingList([2, 4, 6], SEGMENT_SIZE)  # Анти-лист содержит [0, 7, 8, 9, ..., 9999]
result = SearchEngine.execute_and_not(pos_pl, neg_pl)
assert result.doc_ids == [1, 3, 5], f"Expected [1, 3, 5], got {result.doc_ids}"
print("✓ Нет пересечений")


# Тест 4: Полное пересечение - все элементы позитивного списка исключены
print("Тест 4: Полное пересечение")
pos_pl = PostingList([1, 3, 5])
# Создаем анти-лист, который содержит [1, 3, 5] в дополнении
excluded_docs = [0, 2, 4, 6, 7, 8, 9] + list(range(10, SEGMENT_SIZE))
neg_pl = AntiPostingList(excluded_docs, SEGMENT_SIZE)  # Анти-лист содержит [1, 3, 5]
result = SearchEngine.execute_and_not(pos_pl, neg_pl)
assert result.doc_ids == [1, 3, 5], f"Expected [], got {result.doc_ids}"
print("✓ Полное пересечение")



# Тест 5: Частичное пересечение - некоторые элементы остаются
print("Тест 5: Частичное пересечение - некоторые остаются")
pos_pl = PostingList([1, 3, 5, 7])
# Создаем анти-лист, который содержит [1, 3, 5] но НЕ содержит [7]
excluded_docs = [0, 1, 2, 3, 4, 5, 6, 8, 9]
neg_pl = AntiPostingList(excluded_docs, SEGMENT_SIZE)  # Анти-лист содержит [1, 3, 5, 7]
result = SearchEngine.execute_and_not(pos_pl, neg_pl)
assert result.doc_ids == [7], f"Expected [], got {result.doc_ids}"
print("✓ Частичное пересечение - все исключены")


# Тест 6: Случайные данные
print("Тест 6: Случайные данные")
pos_docs = [42, 1337, 9999, 5000, 123]
pos_pl = PostingList(pos_docs)
# Анти-лист исключает [42, 1337, 9999]
excluded_docs = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 42, 1337, 9999]
neg_pl = AntiPostingList(excluded_docs, SEGMENT_SIZE)
result = SearchEngine.execute_and_not(pos_pl, neg_pl)
assert result.doc_ids == [5000, 123], f"Expected [5000, 123], got {result.doc_ids}"
print("✓ Случайные данные")


# Тест 7: Граничные значения
print("Тест 7: Граничные значения")
pos_pl = PostingList([0, 9999])  # Первый и последний элементы сегмента
neg_pl = AntiPostingList([0, 2, 3, 4, 9999], SEGMENT_SIZE)  # Анти-лист содержит [0, 6, 7, ..., 9999]
result = SearchEngine.execute_and_not(pos_pl, neg_pl)
assert result.doc_ids == [], f"Expected [], got {result.doc_ids}"
print("✓ Граничные значения")

# Тест 8: Очень разреженные данные
print("Тест 8: Очень разреженные данные")
pos_docs = [1, 1000, 5000, 9999]
pos_pl = PostingList(pos_docs)
# Анти-лист исключает только [1, 1000]
excluded_docs = [1, 1000] + list(range(5001, 9999))
neg_pl = AntiPostingList(excluded_docs, SEGMENT_SIZE)
result = SearchEngine.execute_and_not(pos_pl, neg_pl)
assert result.doc_ids == [5000, 9999], f"Expected [5000, 9999], got {result.doc_ids}"
print("✓ Очень разреженные данные")

Тест 1: Пустой позитивный список
✓ Пустой позитивный список
Тест 2: Анти-постинг-лист пустой
✓ Анти-постинг-лист пустой
Тест 3: Нет пересечений
✓ Нет пересечений
Тест 4: Полное пересечение
✓ Полное пересечение
Тест 5: Частичное пересечение - некоторые остаются
✓ Частичное пересечение - все исключены
Тест 6: Случайные данные
✓ Случайные данные
Тест 7: Граничные значения
✓ Граничные значения
Тест 8: Очень разреженные данные
✓ Очень разреженные данные


# ...

In [ ]:
print("\n=== Тестирование VirtualNotList ===")
universe = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
excluded = PostingList([2, 4, 6, 8])
not_list = VirtualNotList(universe, excluded)
print(f"Cost: {not_list.cost}")

In [ ]:

# Тестируем поисковый движок
print("=== Тестирование поискового движка ===")

# Создаем тестовые данные
universe = list(range(1, 21))  # Документы 1-20
engine = SearchEngine(universe)

# Добавляем термины
engine.add_term("A", [1, 3, 5, 7, 9, 11])
engine.add_term("B", [2, 4, 6, 8, 10, 12])
engine.add_term("C", [1, 2, 7, 8, 13, 14])
engine.add_term("D", [5, 6, 7, 8, 15, 16])

print("Термины:")
print("A:", [1, 3, 5, 7, 9, 11])
print("B:", [2, 4, 6, 8, 10, 12])
print("C:", [1, 2, 7, 8, 13, 14])
print("D:", [5, 6, 7, 8, 15, 16])
print(f"Universe: {universe}")
print()

# Тестируем различные запросы
test_queries = [
    "A",
    "A AND B",
    "A OR B", 
    "NOT A",
    "A AND NOT B",
    "NOT A AND NOT B",
    "NOT A AND NOT B AND C",
    "A AND B AND C",
    "(A OR B) AND C",
    "NOT (A AND B)"
]

for query in test_queries:
    try:
        result = engine.execute_query(query)
        print(f"'{query}' -> {result}")
    except Exception as e:
        print(f"'{query}' -> ERROR: {e}")
    